In [1]:
# ── RAG BLOCK 1 ───────────────────────────────────
# Title: Setup and Install Dependencies
# Purpose: Mount Drive, create folders, install compatible libraries

from google.colab import drive
drive.mount('/content/drive')

import os

PROJECT = "/content/drive/MyDrive/securescope-ai"

print("Checking folder structure...")

for folder in [
    f"{PROJECT}/data/owasp",
    f"{PROJECT}/models/saved/codebert_binary",
    f"{PROJECT}/src/core",
]:
    os.makedirs(folder, exist_ok=True)
    print(f"✓ {folder}")

print("\nInstalling compatible dependencies...")

# Remove conflicting versions
!pip uninstall -y transformers sentence-transformers peft accelerate -q

# Install compatible versions
!pip install -q \
transformers==4.41.2 \
sentence-transformers==2.7.0 \
accelerate==0.30.1 \
peft==0.10.0 \
sentencepiece \
faiss-cpu \
requests \
beautifulsoup4 \
lxml

print("\nRestarting imports...")

import torch
import transformers
import sentence_transformers
import faiss
import requests
import bs4

print("="*50)
print("VERSIONS")
print("="*50)
print("PyTorch               :", torch.__version__)
print("Transformers          :", transformers.__version__)
print("SentenceTransformers  :", sentence_transformers.__version__)
print("FAISS                 :", faiss.__version__)

if torch.cuda.is_available():
    print("GPU                   :", torch.cuda.get_device_name(0))
else:
    print("GPU                   : CPU")

print("="*50)
print("RAG Block 1 complete.")
print("="*50)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Checking folder structure...
✓ /content/drive/MyDrive/securescope-ai/data/owasp
✓ /content/drive/MyDrive/securescope-ai/models/saved/codebert_binary
✓ /content/drive/MyDrive/securescope-ai/src/core

Installing compatible dependencies...

Restarting imports...
VERSIONS
PyTorch               : 2.11.0+cu128
Transformers          : 4.41.2
SentenceTransformers  : 2.7.0
FAISS                 : 1.14.3
GPU                   : Tesla T4
RAG Block 1 complete.


In [2]:
# ── RAG BLOCK 2 ───────────────────────────────────
# Title: Download Real OWASP Cheat Sheets
# Purpose: Scrape official OWASP documentation
#          Store as plain text for chunking
# Why this approach: real data, not hardcoded text
#                   RAG grounded in actual OWASP docs

import os
import requests
from bs4 import BeautifulSoup

PROJECT = "/content/drive/MyDrive/securescope-ai"
OWASP_DIR = f"{PROJECT}/data/owasp"
os.makedirs(OWASP_DIR, exist_ok=True)

OWASP_URLS = {
    # SQL Injection
    "sql_injection":
        "https://cheatsheetseries.owasp.org/cheatsheets/SQL_Injection_Prevention_Cheat_Sheet.html",

    "query_parameterization":
        "https://cheatsheetseries.owasp.org/cheatsheets/Query_Parameterization_Cheat_Sheet.html",

    "injection_prevention":
        "https://cheatsheetseries.owasp.org/cheatsheets/Injection_Prevention_Cheat_Sheet.html",

    # Command Injection
    "command_injection":
        "https://cheatsheetseries.owasp.org/cheatsheets/OS_Command_Injection_Defense_Cheat_Sheet.html",

    # Secrets / Credentials
    "secrets":
        "https://cheatsheetseries.owasp.org/cheatsheets/Secrets_Management_Cheat_Sheet.html",

    "password_storage":
        "https://cheatsheetseries.owasp.org/cheatsheets/Password_Storage_Cheat_Sheet.html",

    # Path Traversal (replacement for broken URL)
    "path_traversal":
        "https://owasp.org/www-community/attacks/Path_Traversal",

    # Extra sources to improve retrieval
    "input_validation":
        "https://cheatsheetseries.owasp.org/cheatsheets/Input_Validation_Cheat_Sheet.html",

    "file_upload":
        "https://cheatsheetseries.owasp.org/cheatsheets/File_Upload_Cheat_Sheet.html",

    "deserialization":
        "https://cheatsheetseries.owasp.org/cheatsheets/Deserialization_Cheat_Sheet.html",
}

def extract_page_text(url):
    response = requests.get(
        url,
        timeout=30,
        headers={"User-Agent": "SecureScopeAI-RAG/1.0"}
    )
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "lxml")
    for tag in soup(["script", "style", "nav", "footer", "header"]):
        tag.decompose()
    main = soup.find("main") or soup.body
    lines = [
        line.strip()
        for line in main.get_text("\n", strip=True).splitlines()
        if line.strip()
    ]
    return "\n".join(lines)

downloaded = []
failed = []

for name, url in OWASP_URLS.items():
    print(f"Downloading {name}...")
    try:
        text = extract_page_text(url)
        path = f"{OWASP_DIR}/{name}.txt"
        with open(path, "w", encoding="utf-8") as f:
            f.write(text)
        downloaded.append((name, len(text), path))
        print(f"  Saved: {len(text):,} chars")
    except Exception as e:
        failed.append((name, str(e)))
        print(f"  FAILED: {e}")

print(f"\n{'='*50}")
print(f"Downloaded: {len(downloaded)}/{len(OWASP_URLS)}")
for name, chars, path in downloaded:
    print(f"  {name:<30} {chars:>8,} chars")

if failed:
    print(f"\nFailed: {len(failed)}")
    for name, error in failed:
        print(f"  {name}: {error}")

print("\nRAG Block 2 complete.")

  Saved: 17,696 chars
  Saved: 7,317 chars
  Saved: 19,451 chars
  Saved: 10,386 chars
  Saved: 59,212 chars
  Saved: 18,661 chars
  Saved: 6,654 chars
  Saved: 16,996 chars
  Saved: 10,709 chars
  Saved: 17,331 chars

Downloaded: 10/10
  sql_injection                    17,696 chars
  query_parameterization            7,317 chars
  injection_prevention             19,451 chars
  command_injection                10,386 chars
  secrets                          59,212 chars
  password_storage                 18,661 chars
  path_traversal                    6,654 chars
  input_validation                 16,996 chars
  file_upload                      10,709 chars
  deserialization                  17,331 chars

RAG Block 2 complete.


In [3]:
# ── RAG BLOCK 3 ───────────────────────────────────
# Title: Chunk OWASP Documents
# Purpose: Split large text files into manageable
#          chunks for embedding and retrieval
# Why chunking matters:
#   Embedding models have token limits (~512 tokens)
#   Smaller chunks = more precise retrieval
#   Overlap prevents losing context at boundaries

import os
import json

def chunk_text(text: str,
               chunk_size: int = 500,
               overlap: int = 50) -> list:
    """
    Split text into overlapping chunks.

    chunk_size: characters per chunk
    overlap:    characters shared between consecutive chunks
                prevents losing context at boundaries

    Example with overlap:
        chunk 1: "...SQL injection occurs when..."
        chunk 2: "...when user input is concatenated..."
                  ↑ "when" repeated — context preserved
    """
    words = text.split()
    chunks = []
    start = 0

    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        start = end - overlap  # overlap with next chunk

        if start >= len(words):
            break

    return chunks


# Process all downloaded OWASP files
OWASP_DIR = f"{PROJECT}/data/owasp"
all_chunks = []
chunk_metadata = []

print("Chunking OWASP documents...")

for fname in sorted(os.listdir(OWASP_DIR)):
    if not fname.endswith('.txt'):
        continue

    doc_name = fname.replace('.txt', '')
    fpath = os.path.join(OWASP_DIR, fname)

    with open(fpath, 'r', encoding='utf-8') as f:
        text = f.read()

    chunks = chunk_text(
    text,
    chunk_size=200,
    overlap=40
)

    for i, chunk in enumerate(chunks):
        if len(chunk.strip()) < 50:  # skip very short chunks
            continue
        all_chunks.append(chunk)
        chunk_metadata.append({
            'source':   doc_name,
            'chunk_id': i,
            'text':     chunk
        })

    print(f"  {doc_name:<30}: {len(chunks):>3} chunks")

print(f"\nTotal chunks: {len(all_chunks)}")
print(f"Avg chunk length: {sum(len(c) for c in all_chunks) // len(all_chunks)} chars")

# Save metadata
metadata_path = f"{PROJECT}/models/saved/chunk_metadata.json"
with open(metadata_path, 'w', encoding='utf-8') as f:
    json.dump(chunk_metadata, f, indent=2)

print(f"Metadata saved: {metadata_path}")
print("RAG Block 3 complete.")

Chunking OWASP documents...
  command_injection             :  11 chunks
  cryptographic_storage         :  15 chunks
  deserialization               :  17 chunks
  django_security               :  10 chunks
  file_upload                   :  11 chunks
  injection_prevention          :  20 chunks
  input_validation              :  17 chunks
  password_storage              :  19 chunks
  path_traversal                :   6 chunks
  path_traversal_testing        :   6 chunks
  query_parameterization        :   8 chunks
  secrets                       :  58 chunks
  secure_code_review            :  15 chunks
  sql_injection                 :  18 chunks

Total chunks: 230
Avg chunk length: 1257 chars
Metadata saved: /content/drive/MyDrive/securescope-ai/models/saved/chunk_metadata.json
RAG Block 3 complete.


In [4]:
# ── RAG BLOCK 4 ───────────────────────────────────
# Title: Embed Chunks and Build FAISS Index
# Purpose: Convert text chunks to vectors
#          Build searchable similarity index
# Why sentence-transformers over OpenAI embeddings:
#   Free, runs locally, no API key needed
#   all-MiniLM-L6-v2: 80MB, fast, good quality
#   Sufficient for domain-specific OWASP retrieval

import faiss
import numpy as np
import pickle
from sentence_transformers import SentenceTransformer

# Load embedding model
print("Loading embedding model...")
embedder = SentenceTransformer('multi-qa-mpnet-base-dot-v1')
print(f"Model loaded. Embedding dimension: 384")

# Embed all chunks
print(f"\nEmbedding {len(all_chunks)} chunks...")
print("This takes 2-3 minutes...")

embeddings = embedder.encode(
    all_chunks,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)

print(f"Embeddings shape: {embeddings.shape}")

# Normalize for cosine similarity
faiss.normalize_L2(embeddings)

# Build FAISS index
# IndexFlatIP = exact search with inner product
# After normalization, inner product = cosine similarity
dimension = embeddings.shape[1]  # 384
index = faiss.IndexFlatIP(dimension)
index.add(embeddings.astype('float32'))

print(f"FAISS index built: {index.ntotal} vectors")

# Save index and metadata
faiss_path = f"{PROJECT}/models/saved/owasp_faiss.index"
pkl_path   = f"{PROJECT}/models/saved/owasp_metadata.pkl"

faiss.write_index(index, faiss_path)
with open(pkl_path, 'wb') as f:
    pickle.dump(chunk_metadata, f)

print(f"\nFAISS index saved: {faiss_path}")
print(f"Metadata saved:    {pkl_path}")
print(f"Index size: {os.path.getsize(faiss_path) / 1024:.1f} KB")
print("RAG Block 4 complete.")

Loading embedding model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(



/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Model loaded. Embedding dimension: 384

Embedding 230 chunks...
This takes 2-3 minutes...


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Embeddings shape: (230, 768)
FAISS index built: 230 vectors

FAISS index saved: /content/drive/MyDrive/securescope-ai/models/saved/owasp_faiss.index
Metadata saved:    /content/drive/MyDrive/securescope-ai/models/saved/owasp_metadata.pkl
Index size: 690.0 KB
RAG Block 4 complete.


In [5]:
# ── RAG BLOCK 5 ───────────────────────────────────
# Title: Test RAG Retrieval
# Purpose: Verify FAISS retrieves correct chunks
#          Test each of the 5 vulnerability types
# This is the validation step before LLM integration

def retrieve(query: str, top_k: int = 3) -> list:
    """
    Retrieve top_k most relevant OWASP chunks.
    Returns list of dicts with text + source + score.
    """
    q_emb = embedder.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(q_emb)
    scores, indices = index.search(q_emb.astype('float32'), top_k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        results.append({
            'text':   chunk_metadata[idx]['text'],
            'source': chunk_metadata[idx]['source'],
            'score':  round(float(score), 4)
        })
    return results


# Test queries — one per vulnerability type
TYPE_QUERIES = {
    'sql_injection':    'SQL injection parameterized query prepared statement prevent',
    'hardcoded_secret': 'hardcoded credential password secret key environment variable vault',
    'insecure_eval':    'OS command injection shell metacharacter subprocess shlex',
    'path_traversal':   'path traversal directory dot dot slash canonical filename validate',
    'cmd_injection':    'command injection shell subprocess list shlex quote sanitize',
}

# Keep test_queries pointing to same dict for the test block
test_queries = TYPE_QUERIES

print("=== RETRIEVAL TEST ===\n")
all_passed = True

for vtype, query in test_queries.items():
    results = retrieve(query, top_k=3)
    top_source = results[0]['source']
    top_score  = results[0]['score']
    top_text   = results[0]['text'][:100]

    # Check if top result is from relevant source
    relevant_keywords = {
        'sql_injection':    ['sql', 'injection', 'query'],
        'hardcoded_secret': ['secret', 'credential', 'password'],
        'insecure_eval':    ['command', 'injection', 'execution'],
        'path_traversal':   ['file', 'upload', 'path'],
        'cmd_injection':    ['command', 'injection', 'os'],
    }

    text_lower = top_text.lower()
    relevant = any(
        kw in text_lower or kw in top_source
        for kw in relevant_keywords[vtype]
    )

    status = "PASS" if relevant else "CHECK"
    if not relevant:
        all_passed = False

    print(f"[{status}] {vtype}")
    print(f"  Query:  {query}")
    print(f"  Source: {top_source}  Score: {top_score}")
    print(f"  Text:   {top_text}...")
    print()

print(f"All relevant: {all_passed}")
print("RAG Block 5 complete.")

=== RETRIEVAL TEST ===

[PASS] sql_injection
  Query:  SQL injection parameterized query prepared statement prevent
  Source: query_parameterization  Score: 0.7353
  Text:   Query Parameterization Cheat Sheet ¶ Introduction ¶ SQL Injection is one of the most dangerous web v...

[PASS] hardcoded_secret
  Query:  hardcoded credential password secret key environment variable vault
  Source: secrets  Score: 0.5705
  Text:   : vault:latest args : [ "agent" , "-config=/etc/vault/vault-agent-config.hcl" ] volumeMounts : - nam...

[PASS] insecure_eval
  Query:  OS command injection shell metacharacter subprocess shlex
  Source: command_injection  Score: 0.623
  Text:   OS Command Injection Defense Cheat Sheet ¶ Introduction ¶ Command injection (or OS Command Injection...

[PASS] path_traversal
  Query:  path traversal directory dot dot slash canonical filename validate
  Source: path_traversal_testing  Score: 0.5995
  Text:   Path Traversal Overview A path traversal attack (also known as direc

In [6]:
import transformers
import torch

print("Transformers:", transformers.__version__)
print("Torch:", torch.__version__)

Transformers: 4.41.2
Torch: 2.11.0+cu128


In [7]:
import torch

print(torch.cuda.is_available())

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

True
Tesla T4


In [8]:
!pip uninstall -y transformers
!pip install -q transformers==4.41.2 accelerate sentencepiece

Found existing installation: transformers 4.41.2
Uninstalling transformers-4.41.2:
  Successfully uninstalled transformers-4.41.2


In [9]:
import transformers
print(transformers.__version__)

4.41.2


In [10]:
# ── RAG BLOCK 6 ───────────────────────────────────
# Title: Load Phi-3-mini
# Purpose: Load the LLM for explanation generation
# Why Phi-3-mini:
#   3.8B parameters — fits on T4 GPU with float16
#   Microsoft instruction-tuned model
#   Reliable structured output with tight prompts
#   Free, runs locally, no API cost

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

print("Loading Phi-3-mini-4k-instruct...")
print("This takes 5-10 minutes on T4 GPU.")
print("Do not interrupt.\n")

MODEL_ID = "microsoft/Phi-3-mini-4k-instruct"

phi_tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True
)

phi_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

phi_model.eval()

print(f"\nPhi-3-mini loaded successfully.")
print(f"Device: {next(phi_model.parameters()).device}")
print(f"dtype:  {next(phi_model.parameters()).dtype}")

# Quick sanity test
print("\nRunning sanity test...")
test_input = phi_tokenizer(
    "What is SQL injection?",
    return_tensors="pt"
).to(phi_model.device)

with torch.no_grad():
    _ = phi_model.generate(
        **test_input,
        max_new_tokens=10,
        do_sample=False
    )

print("Sanity test passed. Model responding.")
print("RAG Block 6 complete.")

Loading Phi-3-mini-4k-instruct...
This takes 5-10 minutes on T4 GPU.
Do not interrupt.



Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]


Phi-3-mini loaded successfully.
Device: cuda:0
dtype:  torch.float16

Running sanity test...
Sanity test passed. Model responding.
RAG Block 6 complete.


In [13]:
# ── RAG BLOCK 7 ───────────────────────────────────
# Title: Full RAG Pipeline — End to End Test
# Purpose: CodeBERT detects → Feature extractor types
#          → RAG retrieves → Phi-3-mini explains
# This is the complete Phase 3 pipeline

import re as _re

def generate_explanation(code: str,
                         vuln_type: str,
                         top_k_chunks: int = 3) -> dict:
    """
    Complete RAG pipeline for one code sample.

    Steps:
    1. Retrieve relevant OWASP chunks
    2. Build prompt with code + context
    3. Generate structured explanation
    4. Parse DANGER + FIX + REF
    """

    # Step 1: Retrieve OWASP context
    query   = TYPE_QUERIES.get(vuln_type,
              f"{vuln_type.replace('_', ' ')} prevention Python")
    context = retrieve(query, top_k=top_k_chunks)
    context_text = "\n\n".join([
        f"[{c['source']}]\n{c['text']}"
        for c in context
    ])

    # Step 2: Build structured prompt
    prompt = f"""You are a Python security expert reviewing code for vulnerabilities.

VULNERABILITY TYPE: {vuln_type.replace('_', ' ').upper()}

VULNERABLE CODE:
```python
{code}
```

OWASP SECURITY GUIDANCE:
{context_text}

Based on the code and OWASP guidance above, provide exactly:

DANGER: [One sentence — what an attacker can do with this vulnerability]
FIX: [The corrected Python code only, no prose]
REF: [The most relevant OWASP source from the guidance above]"""

    messages = [{"role": "user", "content": prompt}]

    # Step 3: Generate
    inputs = phi_tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(phi_model.device)

    with torch.no_grad():
        outputs = phi_model.generate(
            inputs,
            max_new_tokens=300,
            temperature=0.1,
            do_sample=True,
            pad_token_id=phi_tokenizer.eos_token_id,
            eos_token_id=phi_tokenizer.eos_token_id,
        )

    raw = phi_tokenizer.decode(
        outputs[0][inputs.shape[1]:],
        skip_special_tokens=True
    ).strip()

    # Step 4: Parse structured output
    result = {
        "vuln_type":       vuln_type,
        "danger":          "",
        "fix":             "",
        "ref":             "",
        "raw":             raw,
        "context_sources": [c['source'] for c in context]
    }

    # DANGER — single line before FIX
    danger_match = _re.search(
        r'DANGER:\s*(.+?)(?=\nFIX:|\nREF:|$)', raw, _re.DOTALL
    )
    if danger_match:
        result['danger'] = danger_match.group(1).strip()

    # FIX — may be multi-line code block
    fix_match = _re.search(
        r'FIX:\s*(.*?)(?=\nREF:|$)', raw, _re.DOTALL
    )
    if fix_match:
        fix_raw = fix_match.group(1).strip()
        # Strip markdown code fences if present
        fix_raw = _re.sub(r'^```(?:python)?\n?', '', fix_raw)
        fix_raw = _re.sub(r'\n?```$', '',         fix_raw)
        result['fix'] = fix_raw.strip()

    # REF — single line
    ref_match = _re.search(r'REF:\s*(.+?)(?:\n|$)', raw)
    if ref_match:
        result['ref'] = ref_match.group(1).strip()

    return result


# ── Test all 5 vulnerability types ────────────────
test_cases = [
    {
        "code":      "def get_user(uid):\n    return db.execute('SELECT * FROM users WHERE id = ' + uid)",
        "vuln_type": "sql_injection"
    },
    {
        "code":      "def connect():\n    password = 'admin123'\n    return db.connect(password=password)",
        "vuln_type": "hardcoded_secret"
    },
    {
        "code":      "def calc(expr):\n    return eval(expr)",
        "vuln_type": "insecure_eval"
    },
    {
        "code":      "def read_file(filename):\n    with open('/uploads/' + filename) as f:\n        return f.read()",
        "vuln_type": "path_traversal"
    },
    {
        "code":      "def ping(host):\n    os.system('ping ' + host)",
        "vuln_type": "cmd_injection"
    },
]

print("=" * 60)
print("FULL RAG PIPELINE TEST — All 5 Vulnerability Types")
print("=" * 60)

results = []
for tc in test_cases:
    print(f"\nTesting: {tc['vuln_type']}")
    print(f"Code: {tc['code'][:60]}...")

    result = generate_explanation(tc['code'], tc['vuln_type'])

    print(f"DANGER:  {result['danger']}")
    print(f"FIX:     {result['fix'][:120]}...")
    print(f"REF:     {result['ref']}")
    print(f"Sources: {result['context_sources']}")

    results.append(result)

# Quality check
print("\n" + "=" * 60)
print("QUALITY CHECK")
print("=" * 60)
all_pass = True
for r in results:
    danger_ok = len(r['danger']) > 20
    fix_ok    = len(r['fix'])    > 10
    ref_ok    = len(r['ref'])    >  5
    status    = "PASS" if all([danger_ok, fix_ok, ref_ok]) else "PARTIAL"
    if status == "PARTIAL":
        all_pass = False
    print(f"[{status}] {r['vuln_type']:<20} "
          f"danger={danger_ok} fix={fix_ok} ref={ref_ok}")

print(f"\nAll passed: {all_pass}")
print("RAG Block 7 complete.")

FULL RAG PIPELINE TEST — All 5 Vulnerability Types

Testing: sql_injection
Code: def get_user(uid):
    return db.execute('SELECT * FROM user...
DANGER:  An attacker can execute arbitrary SQL commands, potentially leading to data theft, data manipulation, or unauthorized access to the underlying OS.
FIX:     def get_user(uid):
    query = "SELECT * FROM users WHERE id = %s"
    return db.execute(query, (uid,))...
REF:     [query_parameterization]
Sources: ['query_parameterization', 'injection_prevention', 'query_parameterization']

Testing: hardcoded_secret
Code: def connect():
    password = 'admin123'
    return db.conne...
DANGER:  An attacker can easily gain unauthorized access to the database by using the hardcoded password 'admin123'.
FIX:     import os
from my_database_module import connect_to_db

def connect():
    password = os.getenv('DB_PASSWORD')
    retur...
REF:     [secrets]
Sources: ['secrets', 'secrets', 'secrets']

Testing: insecure_eval
Code: def calc(expr):
    retu

In [14]:
# ── RAG BLOCK 8 ───────────────────────────────────
# Title: Save Complete Phase 3 Pipeline
# Purpose: Save all components with config
#          Ready for FastAPI integration

import json

phase3_config = {
    "version": "phase3_rag_v1",
    "pipeline_steps": [
        "1. CodeBERT binary classifier → is_vulnerable",
        "2. Phase 1 feature extractor → vuln_type",
        "3. FAISS retrieval → top 3 OWASP chunks",
        "4. Phi-3-mini → DANGER + FIX + REF"
    ],
    "components": {
        "binary_classifier": {
            "model":    "codebert_binary/",
            "type":     "HuggingFace AutoModelForSequenceClassification",
            "base":     "microsoft/codebert-base",
            "f1":        0.954,
            "dataset":  "PyCode Vul 14,248 functions"
        },
        "type_identifier": {
            "method":   "rule-based feature extractor",
            "file":     "src/core/feature_extractor.py",
            "types":    ["sql_injection", "hardcoded_secret",
                         "insecure_eval", "path_traversal", "cmd_injection"],
            "rationale": "insufficient labeled type data for ML approach"
        },
        "rag": {
            "embedder":   "multi-qa-mpnet-base-dot-v1",
            "index":      "owasp_faiss.index",
            "metadata":   "owasp_metadata.pkl",
            "sources":    list(OWASP_URLS.keys()),
            "total_chunks": len(all_chunks)
        },
        "llm": {
            "model":       "microsoft/Phi-3-mini-4k-instruct",
            "dtype":       "float16",
            "max_tokens":  250,
            "temperature": 0.1
        }
    }
}

config_path = f"{PROJECT}/models/saved/phase3_config.json"
with open(config_path, 'w') as f:
    json.dump(phase3_config, f, indent=2)

print("=" * 50)
print("PHASE 3 RAG PIPELINE SAVED")
print("=" * 50)
print(f"\nFiles saved to {PROJECT}/models/saved/:")
print(f"  owasp_faiss.index     ← FAISS vector index")
print(f"  owasp_metadata.pkl    ← chunk metadata")
print(f"  chunk_metadata.json   ← readable metadata")
print(f"  phase3_config.json    ← pipeline config")
print(f"\nOWASP docs saved to {PROJECT}/data/owasp/:")
for name, chars, path in downloaded:
    print(f"  {name}.txt")
print(f"\nPhase 3 RAG complete.")
print(f"Next: FastAPI /scan/explain endpoint")

PHASE 3 RAG PIPELINE SAVED

Files saved to /content/drive/MyDrive/securescope-ai/models/saved/:
  owasp_faiss.index     ← FAISS vector index
  owasp_metadata.pkl    ← chunk metadata
  chunk_metadata.json   ← readable metadata
  phase3_config.json    ← pipeline config

OWASP docs saved to /content/drive/MyDrive/securescope-ai/data/owasp/:
  sql_injection.txt
  query_parameterization.txt
  injection_prevention.txt
  command_injection.txt
  secrets.txt
  password_storage.txt
  path_traversal.txt
  input_validation.txt
  file_upload.txt
  deserialization.txt

Phase 3 RAG complete.
Next: FastAPI /scan/explain endpoint
